# Understanding LangChain Chains for Large Language Model Application Development

Complete updated companion, based on the author-supplied LangChain book notebook. Run top to bottom with a locally configured OpenAI key. Calls incur API charges. The career Markdown is an original teaching fixture, not the draft ebook. Original archive code and transcripts remain in the nonexecuting appendix.

## Notebook setup

In [ ]:
import os,sys,subprocess,importlib.metadata,hashlib,urllib.request
from pathlib import Path
from tempfile import TemporaryDirectory
dependencies = {'pandas': '2.3.3', 'numpy': '2.3.3', 'openai': '2.43.0', 'langchain': '1.2.12', 'langchain-openai': '1.1.11', 'langchain-core': '1.4.8', 'langgraph': '1.1.2', 'langchain-community': '0.4.1', 'langchain-text-splitters': '1.1.1', 'ddgs': '9.14.4', 'python-dotenv': '1.2.2', 'pydantic': '2.13.4', 'httpx': '0.28.1', 'certifi': '2025.11.12'}
missing = []
for name,version in dependencies.items():
    try:
        if importlib.metadata.version(name) != version: missing.append(f'{name}=={version}')
    except importlib.metadata.PackageNotFoundError:
        missing.append(f'{name}=={version}')
if missing:
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
import certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Set OPENAI_API_KEY locally before running this notebook. Never paste it into a cell.')
os.environ.setdefault('MODEL','gpt-5.4-mini')
os.environ.setdefault('EMBEDDING_MODEL','text-embedding-3-small')
workspace = TemporaryDirectory(prefix='tdb-langchain-12-')
work = Path(workspace.name)
source_revision = '5984620e5691a3991d5e9353bc2cf2533e576234'
files = {'product-reviews.csv': 'ce8895ea5d502554d8da2c7fc6b20e6e9591816ee96e66a8ad329bec7c009522', 'career-in-ai.md': 'bbe0f834b947e46a5e4d152119ee76bb7ed784820d7e057c06c8f1e3b0a3971c'}
for name,digest in files.items():
    url = f'https://raw.githubusercontent.com/youssefHosni/hands-on-langchain/{source_revision}/data/{name}'
    with urllib.request.urlopen(url,timeout=60) as response: payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == digest, name
    (work/name).write_bytes(payload)
print('Python:',sys.version.split()[0])
print('Configured chat model:',os.environ['MODEL'])
print('Pinned example CSV and Markdown inputs verified; no draft ebook is used.')

## Working environment

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv(find_dotenv(usecwd=True))
llm_model = os.getenv("MODEL", "gpt-5.4-mini")

In [ ]:
df = pd.read_csv(work / "product-reviews.csv")
df.head()

## A basic chain

The most fundamental "chain" combines a **prompt** and a **model**. In LangChain 1.x you build it with
the pipe operator (`|`) — this is the LangChain Expression Language (LCEL). We add `StrOutputParser()`
so the chain returns a plain string instead of a message object.

We use a high temperature here to get more diverse, creative answers.

In [ ]:
llm = ChatOpenAI(temperature=0.9, model=llm_model, max_tokens=2048, max_retries=0, reasoning_effort="none")

prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)

chain = prompt | llm | StrOutputParser()

In [ ]:
product = "Deep Learning GPUs"
chain.invoke({"product": product})

## Sequential chains

Often you want to run several steps in order, feeding the output of one into the next.

### A simple two-step sequence

Chain one turns a product into a company name; chain two writes a short description of that company.
We connect them by mapping chain one's string output into the `{company_name}` slot that chain two
expects.

In [ ]:
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe a company that makes {product}?"
)
second_prompt = ChatPromptTemplate.from_template(
    "Write a 30 word description for the following company: {company_name}"
)

# chain one produces the name; we feed it into chain two as `company_name`
overall_simple_chain = (
    {"company_name": first_prompt | llm | StrOutputParser()}
    | second_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
overall_simple_chain.invoke({"product": product})

### A multi-step sequence with several inputs and outputs

What if later steps need *several* earlier results, and you want to keep all the intermediate values?
The idiom for this is `RunnablePassthrough.assign`, which runs a sub-chain and **adds its result as a
new key** to the data flowing through — so every later step can read every earlier result by name.

We will take a (possibly non-English) product review and, in sequence:

1. translate it to English (`English_Review`),
2. summarize it in one sentence (`summary`),
3. detect its original language (`language`),
4. write a follow-up reply in that original language (`followup_message`).

In [ ]:
from langchain_core.runnables import RunnablePassthrough

translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:\n\n{Review}"
)
summary_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:\n\n{English_Review}"
)
language_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
followup_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)

# each .assign adds one key; later prompts read keys produced earlier
overall_chain = (
    RunnablePassthrough.assign(English_Review=translate_prompt | llm | StrOutputParser())
    .assign(summary=summary_prompt | llm | StrOutputParser())
    .assign(language=language_prompt | llm | StrOutputParser())
    .assign(followup_message=followup_prompt | llm | StrOutputParser())
)

In [ ]:
review = df["Review"].iloc[5]
result = overall_chain.invoke({"Review": review})
result

## Routing between chains

Sometimes you want to send the input to a *different* specialized chain depending on what it is — a
physics question to a physics expert, a math question to a mathematician, and so on. This is **routing**.

In LangChain 1.x the clean way to route is:

1. a small **classifier** chain that decides the topic (we force a fixed set of labels with
   `with_structured_output`), then
2. a plain Python function that picks the matching destination chain.

First, the four specialist prompts and their chains.

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{input}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. You have an excellent knowledge of and \
understanding of people, events and contexts from a range of historical periods.

Here is a question:
{input}"""

cs_template = """You are a successful computer scientist. You are great at answering coding \
questions because you can describe a solution in clear imperative steps and balance time and \
space complexity.

Here is a question:
{input}"""

llm0 = ChatOpenAI(temperature=0, model=llm_model, max_tokens=2048, max_retries=0, reasoning_effort="none")

destination_chains = {
    "physics": ChatPromptTemplate.from_template(physics_template) | llm0 | StrOutputParser(),
    "math": ChatPromptTemplate.from_template(math_template) | llm0 | StrOutputParser(),
    "history": ChatPromptTemplate.from_template(history_template) | llm0 | StrOutputParser(),
    "computer science": ChatPromptTemplate.from_template(cs_template) | llm0 | StrOutputParser(),
}
# default chain for anything that doesn't fit a specialist
default_chain = ChatPromptTemplate.from_template("{input}") | llm0 | StrOutputParser()

Now the classifier. We use a Pydantic schema with a constrained `Literal` field, so the model can
only return one of our known destinations (or `DEFAULT`).

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    """Pick the best destination for the user's question."""
    destination: Literal["physics", "math", "history", "computer science", "DEFAULT"] = Field(
        description="The subject best suited to answer the question; DEFAULT if none fit."
    )

router_prompt = ChatPromptTemplate.from_template(
    "Given the user question below, classify it as one of: physics, math, history, "
    "computer science, or DEFAULT.\n\nQuestion: {input}"
)
router = router_prompt | llm0.with_structured_output(RouteQuery)

Finally, a routing function ties it together: classify, then invoke the chosen chain.

In [ ]:
from langchain_core.runnables import RunnableLambda

def route(info):
    destination = router.invoke({"input": info["input"]}).destination
    chain = destination_chains.get(destination, default_chain)
    print(f"[router] -> {destination}")
    return chain.invoke({"input": info["input"]})

full_chain = RunnableLambda(route)

In [ ]:
print(full_chain.invoke({"input": "What is the speed of light?"}))

In [ ]:
print(full_chain.invoke({"input": "What is 2 + 2 times 8?"}))

## Regression checks

These checks verify data flow and contracts. Explicit scripted probes test control flow only; they are separate from the live model calls above.

In [ ]:
import json
assert len(df)==6 and 'Je trouve' in review
assert set(result) == {'Review','English_Review','summary','language','followup_message'}
assert result['Review'] == review and all(isinstance(result[k],str) and result[k] for k in result)
for destination in ['physics','math','history','computer science','DEFAULT']:
    assert RouteQuery(destination=destination).destination == destination
try:
    RouteQuery(destination='unknown')
except ValueError: pass
else: raise AssertionError('Unexpected route accepted')
from langchain_core.runnables import RunnableLambda
# Explicit scripted integration probes, not a substitute for the live calls above.
probe = (RunnablePassthrough.assign(first=RunnableLambda(lambda x: x['Review'].upper()))
         .assign(second=RunnableLambda(lambda x: x['first']+'!')))
assert probe.invoke({'Review':'abc'}) == {'Review':'abc','first':'ABC','second':'ABC!'}
print(json.dumps({'source_rows':len(df),'sequence_keys':sorted(result),'route_labels':5,'checks':'passed'}))

## Original archive appendix

The following is the complete original article text and all its original code/transcripts, preserved for reference. Fenced examples here are not executed. The maintained tutorial above follows the newer author-supplied notebook.

Hands-On LangChain for LLMs App Development: Chains

One of the fundamental pillars of LangChain, as implied by its name, is the concept of “chains.” These chains typically integrate a large language model (LLM) with a prompt.

Through these chain structures, you have the ability to assemble multiple building blocks, enabling the execution of a series of operations on your text or other data.

This article will delve into the significance of these chains, ranging from basic forms like the Simple Sequential Chain to more sophisticated variations such as the Router Chain, elucidated with practical illustrations.

## Setting Up Working Environment & Getting Started

We will start by loading the environment variables and the LLM that we will use in this article. Then we’re also going to load the data examples that we’re going to use.

````python
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

llm_model = "gpt-3.5-turbo"
````

One of the capabilities of chains is that you can run them over many inputs at a time. Here we’re going to load a pandas data frame. Pandas data frame is just a data structure that contains a bunch of different elements of data. The main point here is that we’re loading some data that we can then use later on.

````python
import pandas as pd
df = pd.read_csv('Data.csv')
````

If we look inside this pandas data frame, we can see that there is a product column and then a review column. Each of these rows is a different data point that we can start passing through our chains.

````python
df.head()
````

Original figure: https://cdn-images-1.medium.com/max/800/1*vKlw0x45CQhpI4hCZrgoAA.png

The first chain we’re going to cover is the LLM chain. This is a simple but really powerful chain that underpins a lot of the chains that we’ll go over in the future.

## LLM Chain

We will start with importing three different things. We’re going to import the OpenAI model, the chat prompt, and the LLMChain.

````python
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
````

Let's initialize the language model that we want to use. So we’re gonna initialize the chat open AI with a temperature, with a high temperature so that we can get diverse answers.

````python
llm = ChatOpenAI(temperature=0.9, model=llm_model)
````

Now we will initialize a prompt and this prompt is going to take in a variable called product. It’s gonna ask the LLM to generate the best name to describe a company that makes that product.

````python
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)
````

Finally, we’re going to combine these two things into a chain. This is what we call an LLM chain. And it’s quite simple. It’s just the combination of the LLM and the prompt. But now this chain will let us run through the prompt and into the LLM sequentially.

````python
chain = LLMChain(llm=llm, prompt=prompt)
````

If we have a product called Deep Learning GPUs , we can run this through the chain by using chain.run() . This will do is it will format the prompt under the hood, and then it will pass the whole prompt into the LLM. We can see that we get back the name of this hypothetical company called DeepTech GPUs.

````python
product = "Deep Learning GPUs"
chain.run(product)
````

````text
‘DeepTech GPUs.’
````

Here would be a good time to try it, you can input any product descriptions that you would want, and you can see what the chain will output as a result.

So the LLM chain is the most basic type of chain, and that’s gonna be used a lot in the future. And so we can see how this will be used in the next type of chain, which will be sequential chains.

## Sequential Chains

Sequential chains run a sequence of chains, one after another. So to start, you’re going to import the simple sequential chain. This works well when we have subchains that expect only one input and return only one output.

````python
from langchain.chains import SimpleSequentialChain
````

### Simple Sequential Chain

We are going to first create one chain, which uses an LLM and a prompt. This prompt is going to take in the product and will return the best name to describe that company. So that will be the first chain.

````python
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)
````

Next, we’re going to create a second chain. The second chain will take in the company name and then output a 30-word description of that company.

````python
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 30 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)
````

You can imagine how these chains might want to be run one after another, where the output of the first chain, the company name is then passed into the second chain. We can easily do this by creating a simple sequential chain where we have the two chains described there. We’ll call this overall simple chain.

````python
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True)
````

Now, we will run this chain over any product description. And so if we use it with the product above, the Deep Learning GPUs , we can run it over and we can see that it first outputs royal bedding, and then it passes it into the second chain and it comes up with this description of what that company could be about.

````python
overall_simple_chain.run(product)
````

````text
Entering new SimpleSequentialChain chain…
NeuralTech
NeuralTech is a technology company specializing in artificial intelligence and machine learning solutions for businesses across various industries.
````

````text
> Finished chain.
‘NeuralTech is a technology company specializing in artificial intelligence and machine learning solutions for businesses across various industries.’
````

The simple sequential chain works well when there’s only a single input and a single output. But what about when there are multiple inputs or multiple outputs? And so we can do this by using just the regular sequential chain.

### Complex Sequential Chain

So let’s import the sequential chain, and then you’re going to create a bunch of chains that we’re going to use one after another.

````python
from langchain.chains import SequentialChain
````

We’re going to be using the data from above, which has a review. In the first chain, we’re going to take the review and translate it into English.

````python
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )
````

With the second chain, we’re going to create a summary of that review in one sentence. This will use the previously generated English review.

````python
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )
````

The third chain is going to detect what the language of the review was in the first place. And so if you notice, this is using the review variable that is coming from the original review.

````python
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )
````

Finally, the fourth chain will take in multiple inputs. So this will take in the summary variable, which we calculated with the second chain, and the language variable, which we calculated with the third chain. And it’s going to ask for a follow-up response to the summary in the specified language.

````python
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )
````

One important thing to note about all these subchains is that the input keys and output keys need to be pretty precise. So here we’re taking in review. This is a variable that will be passed in at the start. We can see that we explicitly set the output key to English review.

This is then used in the next prompt down below, where we take an English review with that same variable name. And we set the output key of that chain to the summary, which we can see is used in the final chain.

The third prompt takes in review, the original variable, and outputs language, which is again used in the final prompt. It’s really important to get these variable names lined up exactly right because there are so many different inputs and outputs going on. And if you get any key errors, you should check that they are lined up. The simple sequential chain takes in multiple chains where each one has a single input and a single output.

This is useful when you have more complicated downstream chains that need to be a composition of multiple previous chains. Now that we have all these chains, we can easily combine them into the sequential chain. So you’ll notice here that we’ll pass in the four chains we created into the chains variable. We’ll create the input variable with the one human input, which is the review. And then we want to return all of the intermediate outputs. So the English review, the summary, and then the follow-up message.

````python
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["English_Review", "summary","followup_message"],
    verbose=True
)
````

Now we can run this over some of the data. So let’s choose a review and pass it in through the overall chain.

````python
review = df.Review[5]
overall_chain(review)
````

````text
> Entering new SequentialChain chain…
````

````text
> Finished chain.
{‘Review’: “Je trouve le goût médiocre. La mousse ne tient pas, c’est bizarre. J’achète les mêmes dans le commerce et le goût est bien meilleur…\nVieux lot ou contrefaçon !?”,
 ‘English_Review’: “I find the taste mediocre. The foam doesn’t hold, it’s weird. I buy the same ones in stores and the taste is much better… Old batch or fake!?”,
 ‘summary’: ‘The reviewer finds the taste of the product mediocre and suspects that it might be an old batch or fake as the foam does not hold.’,
 ‘followup_message’: “Résumé: Le critique trouve le goût du produit médiocre et soupçonne qu’il pourrait s’agir d’un vieux lot ou d’une contrefaçon car la mousse ne tient pas.\n\nRéponse : Le critique a exprimé sa déception quant au goût du produit. Nous prenons très au sérieux les commentaires de nos clients et nous aimerions avoir plus d’informations sur le lot en question afin de pouvoir enquêter sur ce problème. Nous garantissons que tous nos produits sont authentiques et qu’ils sont fabriqués avec les normes les plus élevées de qualité. Veuillez nous contacter directement pour que nous puissions résoudre ce problème et vous fournir une expérience de dégustation satisfaisante.”}
````

We can see here that the original review looks like it was in French. We can see the English review as a translation. We can see a summary of that review, and then we can see a follow-up message in the original language of French.

## Router Chain

So far, we’ve covered the LLM chain and then a sequential chain. But what if you want to do something more complicated? A pretty common but basic operation is to route an input to a chain depending on what exactly that input is.

A good way to imagine this is if you have multiple subchains, each of which is specialized for a particular type of input, you could have a router chain, which first decides which subchain to pass it to and then passes it to that chain.

For a concrete example, let’s look at where we are routing between different types of chains depending on the subject that seems to come in. So we have here different prompts. One prompt is good for answering physics questions. The second prompt is good for answering math questions, the third for history, and then a fourth for computer science. Let’s define all these prompt templates.

````python
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""
````

After we have these prompt templates, we can then provide more information about them. We can give each one a name and then a description. This information is going to be passed to the router chain, so the router chain can decide when to use this subchain.

````python
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]
````

Let’s now import the other types of chains that we need.

````python
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate
````

Here we need a multi-prompt chain. This is a specific type of chain that is used when routing between multiple different prompt templates. As you can see, all the options we have are prompt templates themselves. But this is just one type of thing that you can route between. You can route between any type of chain.

The other classes that we’ll implement here are an LLM router chain. This uses a language model itself to route between the different subchains. This is where the description and the name provided above will be used. We’ll also import a router output parser. This parses the LLM output into a dictionary that can be used downstream to determine which chain to use and what the input to that chain should be.

Now we can get around to using it. First, let’s import and define the language model that we will use.

````python
llm = ChatOpenAI(temperature=0, model=llm_model)
````

We now create the destination chains. These are the chains that will be called by the router chain.

````python
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)
````

As you can see, each destination chain itself is a language model chain, an LLM chain. In addition to the destination chains, we also need a default chain. This is the chain that’s called when the router can’t decide which of the sub-chains to use. For the example above, this might be called when the input question has nothing to do with physics, math, history, or computer science.

````python
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)
````

Now we define the template that is used by the LLM to route between the different chains. This has instructions on the task to be done, as well as the specific formatting that the output should be in.

````python
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""
````

Let’s put a few of these pieces together to build the router chain. First, we create the full router template by formatting it with the destinations that we defined above. This template is flexible to a bunch of different types of destinations.

````python
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""
````

Next, we create the prompt template from this template, and then we create the router chain by passing in the LLM and the overall router prompt.

````python
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)
````

Note that here we have the router output parser. This is important as it will help this chain decide which subchains to route between. Finally, we will put it all together, we can create the overall chain. This has a router chain, which is defined here. It has destination chains, which we pass in here. Then we also pass in the default chain.

````python
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )
````

We can now use this chain. So let’s ask it some questions. We will ask it a physics question, we should hopefully see that it is routed to the physics chain with the input, what is blackbody radiation? And then that is passed into the chain down below, and we can see that the response is very detailed with lots of physics details.

````python
chain.run("What is black body radiation?")
````

````text
> Entering new MultiPromptChain chain…
physics: {‘input’: ‘What is black body radiation?’}
> Finished chain.
“Black body radiation refers to the electromagnetic radiation emitted by a perfect black body, which is an object that absorbs all radiation that falls on it and emits radiation at all wavelengths. The radiation emitted by a black body depends only on its temperature and follows a specific distribution known as Planck’s law. This type of radiation is important in understanding the behavior of stars, as well as in the development of technologies such as incandescent light bulbs and infrared cameras.”
````